# Link Check: Do the Report Links Still Work?

## Purpose

The extraction pipeline gives each report to the model as a web link, and the model's service downloads the PDF from the Coroners Court website. A report that the court withdraws, or a link that was copied wrongly, therefore breaks every trial on that report (as happened to report 15, which the court withdrew). This notebook opens every link in `report_links.jsonl` from this computer and reports, for each one, whether it still leads to a PDF and whether that PDF contains the expected file number.

## What each check means

| Status | Meaning |
|---|---|
| `ok` | The link returned a PDF. |
| `missing` | The court returned "not found" (HTTP 404 or 410): the report has been removed, or the link is wrong. |
| `not_pdf` | The link returned a normal answer that is not a PDF (usually an error page shown with a success code). |
| `blocked` | The court refused the request (HTTP 401, 403 or 429). The report may still exist; open the link in a browser to check. |
| `server_error` | The court's server reported a fault (HTTP 5xx). Try again later. |
| `no_connection` | The request did not reach the server (no internet, timeout or a malformed link). |

For links with the status `ok`, the notebook also reads the first pages of the PDF and looks for the file number registered for the report (column `file_number_in_pdf`: `yes`, `no`, `unreadable` or `not checked`). A `no` means that the link opens a PDF, but possibly the wrong one. The notebook also lists links that are shared by more than one report.

## Limits of the check

- The check is made from this computer. The model's service downloads the report from its own servers, so a link that opens here is very likely, but not guaranteed, to work for the model. A link that is `missing` here will fail for the model as well.
- Nothing is sent to the model, no reference standard is read, and no study result is touched. The notebook only opens the links, so it can be run on all sets at any time.
- Links are registered in `data/report_links.jsonl` (one line per report). A report that has been withdrawn carries `"status": "withdrawn"` in that file.

## Requirements

Python 3.11.15 and `pandas` 3.0.3 (the versions used in the study). The file-number check also needs `pypdf` (`pip install pypdf`); without it the links are still checked and the column `file_number_in_pdf` reads `not checked`.

## 1. Settings

In [1]:
from pathlib import Path
from datetime import datetime
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from io import BytesIO
import json, re, time
import pandas as pd

LINKS_FILE = Path("../data/report_links.jsonl")          # one line per report: report_no, dataset, file_number, file_url
OUTPUT_DIR = Path("../data")                  # each run is saved here as a dated CSV file
DATASETS_TO_CHECK = None                          # None = every line of the file; or a list such as ["prompt_development"]
CHECK_FILE_NUMBER = True                          # open each PDF and look for the registered file number (needs pypdf)
PAGES_TO_READ = 5                                 # the file number is on the first pages of a findings document
TIMEOUT_SECONDS = 30
PAUSE_SECONDS = 0.5                               # pause between requests, to limit the load on the court's server
MAX_BYTES = 40_000_000                            # a PDF larger than this is cut off (findings are about 0.3 to 3 MB)
USER_AGENT = "Mozilla/5.0 (compatible; dissertation-link-check)"   # some sites refuse requests that give no browser name

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None
    print("pypdf is not installed: links are checked, but file numbers are not (pip install pypdf).")

## 2. Reading the links

`load_links` reads the file, one JSON object per line. Blank lines are ignored.

In [2]:
def load_links(path: Path, datasets: list[str] | None = None) -> pd.DataFrame:
    """Read the registered links.

    Args:
        path: The JSON Lines file with one report per line.
        datasets: Names of the sets to keep, or None to keep every line.

    Returns:
        One row per line, with the columns report_no, dataset, file_number, file_url,
        registered_status ("active" unless the line says otherwise) and note.
    """
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    df = pd.DataFrame(rows)
    for column, default in [("status", "active"), ("note", "")]:
        if column not in df:
            df[column] = default
        df[column] = df[column].fillna(default)
    df = df.rename(columns={"status": "registered_status"})
    if datasets is not None:
        df = df[df["dataset"].isin(datasets)]
    return df.reset_index(drop=True)


links = load_links(LINKS_FILE, DATASETS_TO_CHECK)
print(f"{len(links)} link(s) to check")
print(links["dataset"].value_counts().to_string())

54 link(s) to check
dataset
external_validation    40
prompt_development      6
internal_validation     5
few_shot_example        2
test_report             1


## 3. Checking one link

`check_link` opens the link and sorts the answer into one of the statuses in the table above. A successful answer must start with `%PDF`, which is how every PDF file begins. A normal answer that is not a PDF is `not_pdf`, because an error page shown with a success code would otherwise look like a working link.

In [3]:
def check_link(url: str) -> dict:
    """Open one link and describe what came back.

    Args:
        url: The web address of the report.

    Returns:
        A dict with status, http_code, content_type, size_kb, final_url, detail and
        the bytes of the answer under the key `body` (empty unless the answer was a PDF).
    """
    result = dict(status="", http_code=None, content_type="", size_kb=None, final_url=url, detail="", body=b"")
    try:
        with urlopen(Request(url, headers={"User-Agent": USER_AGENT}), timeout=TIMEOUT_SECONDS) as response:
            result.update(http_code=response.status, content_type=response.headers.get("Content-Type", ""), final_url=response.geturl())
            body = response.read(MAX_BYTES)
    except HTTPError as error:                                   # the server answered with an error code
        result.update(http_code=error.code, content_type=error.headers.get("Content-Type", "") if error.headers else "")
        if error.code in (404, 410):
            result["status"] = "missing"
        elif error.code in (401, 403, 429):
            result["status"] = "blocked"
        elif error.code >= 500:
            result["status"] = "server_error"
        else:
            result["status"] = "no_connection"
        result["detail"] = f"HTTP {error.code} {error.reason}"
        return result
    except (URLError, TimeoutError, ValueError, OSError) as error:   # no answer at all, or a malformed link
        result.update(status="no_connection", detail=str(error)[:150])
        return result

    result["size_kb"] = round(len(body) / 1024)
    if body.startswith(b"%PDF"):
        result.update(status="ok", body=body)
    else:
        title = re.search(rb"<title[^>]*>(.*?)</title>", body[:20000], re.I | re.S)
        result.update(status="not_pdf", detail=("page title: " + title.group(1).decode("utf-8", "ignore").strip()[:100]) if title else "the answer is not a PDF")
    return result

### Finding the file number inside the PDF

The registered file number (for example `2022/00391153`) is searched for in the text of the first pages. The search ignores leading zeros and spaces around the slash, because a document may print the same number as `2022/391153` or `2022 / 00391153`.

In [4]:
def file_number_in_pdf(pdf_bytes: bytes, file_number: str) -> str:
    """Look for a file number in the first pages of a PDF.

    Args:
        pdf_bytes: The PDF file.
        file_number: The registered file number, for example "2022/00391153".

    Returns:
        "yes" or "no" if the text could be read, "unreadable" if the PDF has no
        text layer or could not be opened, "not checked" if pypdf is not installed.
    """
    if PdfReader is None:
        return "not checked"
    try:
        pages = PdfReader(BytesIO(pdf_bytes)).pages[:PAGES_TO_READ]
        text = " ".join((page.extract_text() or "") for page in pages)
    except Exception:
        return "unreadable"
    if len(text.strip()) < 50:
        return "unreadable"
    year, _, number = file_number.partition("/")
    pattern = rf"{year}\s*[/\-]\s*0*{number.lstrip('0')}\b"
    return "yes" if re.search(pattern, text) else "no"

## 4. Running the check

Every link is opened once, with a short pause between requests. The PDF itself is not saved.

In [5]:
rows = []
for _, link in links.iterrows():
    answer = check_link(link["file_url"])
    in_pdf = file_number_in_pdf(answer["body"], link["file_number"]) if (CHECK_FILE_NUMBER and answer["status"] == "ok") else "not checked"
    rows.append({"report_no": link["report_no"], "dataset": link["dataset"], "file_number": link["file_number"],
                 "status": answer["status"], "http_code": answer["http_code"], "file_number_in_pdf": in_pdf,
                 "size_kb": answer["size_kb"], "detail": answer["detail"], "registered_status": link["registered_status"],
                 "file_url": link["file_url"], "final_url": answer["final_url"]})
    print(f"  {str(link['report_no']):>5}  {link['dataset']:<20} {answer['status']:<14} {in_pdf if answer['status'] == 'ok' else answer['detail']}")
    time.sleep(PAUSE_SECONDS)

results = pd.DataFrame(rows)

      1  prompt_development   ok             yes
      9  prompt_development   ok             yes
     15  prompt_development   not_pdf        page title: 404 | Coroners Court of New South Wales
     19  prompt_development   ok             yes
     39  prompt_development   ok             yes
     51  prompt_development   ok             yes
      2  internal_validation  ok             yes
     14  internal_validation  ok             yes
     16  internal_validation  ok             yes
     29  internal_validation  ok             yes
     48  internal_validation  ok             yes
      3  external_validation  ok             yes
      4  external_validation  ok             yes
      5  external_validation  ok             yes
      6  external_validation  ok             yes
      7  external_validation  ok             yes
      8  external_validation  ok             yes
     10  external_validation  ok             yes
     11  external_validation  ok             yes
     12  external_val

## 5. Results

A link **needs attention** when:

- its status is not `ok` (unless the report is registered as withdrawn, which is expected to fail);
- it is registered as withdrawn but now works (the report may have been put back);
- the PDF does not contain the registered file number (`file_number_in_pdf` is `no`); or
- the same link is registered for more than one report (the pipeline would then process one PDF under two report numbers).

In [6]:
shared = results.groupby("file_url")["report_no"].transform(lambda s: ", ".join(str(n) for n in s) if len(s) > 1 else "")
results["shared_with"] = shared
withdrawn = results["registered_status"] == "withdrawn"
results["needs_attention"] = (
    (~withdrawn & (results["status"] != "ok"))
    | (withdrawn & (results["status"] == "ok"))
    | (results["file_number_in_pdf"] == "no")
    | (results["shared_with"] != "")
)

print(f"Checked {len(results)} link(s) on {datetime.now():%d %B %Y at %H:%M}")
print(results["status"].value_counts().to_string(), "\n")
problems = results[results["needs_attention"]]
if problems.empty:
    print("All links work and match their file numbers.")
else:
    print(f"{len(problems)} link(s) need attention:")
problems[["report_no", "dataset", "file_number", "status", "file_number_in_pdf", "shared_with", "registered_status", "detail"]]

Checked 54 link(s) on 24 October 2026 at 12:27
status
ok         53
not_pdf     1 

All links work and match their file numbers.


,report_no,dataset,file_number,status,file_number_in_pdf,shared_with,registered_status,detail


## 6. Saving the result

The table is saved as a dated CSV file in `data/`, so that earlier checks are kept and a change in the status of a report can be seen.

In [7]:
OUTPUT_DIR.mkdir(exist_ok=True)
saved_to = OUTPUT_DIR / f"link_check_{datetime.now():%Y%m%d_%H%M}.csv"
results.to_csv(saved_to, index=False)
print(f"Saved to {saved_to}")

Saved to link_checks/link_check_20261024_1227.csv
